# Unconfirmed Spectra — Review Queue

The confirmed pipeline (`proposal_mvp.ipynb`) only operates on 1,631 manually annotated spectra.
The remaining **5,899 unannotated spectra** have never been queried against MassWiki.

This notebook:
1. Fetches library hits for all unannotated spectra
2. Scores them using the fitted parameters from the annotated pipeline
3. Flags cases most worth a chemist's review

**Flag categories:**
- `HIGH_CONF_NOVEL` — high posterior on a compound never seen in BinBase (model confident, biologically unexpected)
- `NEAR_TIE` — top two candidates within 0.05 posterior of each other (RT or manual review would resolve)
- `RICH_NO_HIT` — high spectral entropy (H > 1) but model abstains (potentially novel compound)
- `MS1_RESCUE` — low entropy similarity but model is confident (RT/MS1 doing the work, needs verification)

**Fitted parameters reused from annotated pipeline:**
- σ_M = 4.21 ppm
- σ_RT_anno = 14.1 s
- σ_RT_ref = 19.1 s
- α = 10.0
- P_novel = estimated fresh from full spectra set
- Prior = same BinBase quality-weighted prior

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem.inchi import MolToInchi, InchiToInchiKey
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print('WARNING: rdkit not available')

ROOT     = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, 'data')
CODE_DIR = os.path.join(ROOT, 'code')
OUT_DIR  = os.path.join(ROOT, 'out_unconfirmed')
os.makedirs(OUT_DIR, exist_ok=True)

SPECTRA_PATH      = os.path.join(DATA_DIR, 'ttof+neg+hilic.csv')
CONFIRMED_HITS    = os.path.join(DATA_DIR, 'hilic_ttof_neg_masswiki_hits_new.csv')
UNCONFIRMED_HITS  = os.path.join(DATA_DIR, 'hilic_ttof_neg_masswiki_hits_unconfirmed.csv')
BINBASE_PATH      = os.path.join(DATA_DIR, 'hilic_ttof_all.csv')

# ── Fitted parameters from confirmed pipeline ──────────────────────────────
SIGMA_M        = 4.21    # ppm — TTOF mass accuracy
SIGMA_RT_ANNO  = 14.1    # s   — annotation library (real measured RT)
SIGMA_RT_REF   = 19.1    # s   — reference library (Retip 2.0 predicted RT)
ALPHA          = 10.0    # MS2 complexity weight
SIGMA_BROAD    = 50.0    # ppm — null mass distribution
P_RT_NULL      = norm.pdf(0, 0, 300)  # RT null (300 s broad)
NOTA_THRESH    = 0.6     # abstention threshold

print('Paths and parameters set.')
print(f'  σ_M={SIGMA_M} ppm | σ_RT_anno={SIGMA_RT_ANNO}s | σ_RT_ref={SIGMA_RT_REF}s | α={ALPHA}')

---
## 1. Identify Unconfirmed Spectra

In [ ]:
spectra_all = pd.read_csv(SPECTRA_PATH, low_memory=False)

# Confirmed = manually annotated, not yy/zz prefix
confirmed_mask = spectra_all['is_manual_annotated'].fillna(False).astype(bool)
name_ok_mask   = ~spectra_all['name'].astype(str).str.lower().str.startswith(('yy', 'zz'))
confirmed_wids = set(spectra_all[confirmed_mask & name_ok_mask]['wiki_id'].astype(str))

# Unconfirmed = everything else
unconfirmed = spectra_all[
    ~spectra_all['wiki_id'].astype(str).isin(confirmed_wids)
][['wiki_id', 'rt', 'precursor_mz', 'entropy']].copy()

print(f'Total spectra:          {len(spectra_all):,}')
print(f'Annotated:              {len(confirmed_wids):,}')
print(f'Unannotated to query:   {len(unconfirmed):,}')
print()
print('Entropy distribution (unannotated):')
print(unconfirmed['entropy'].describe().round(3))

---
## 2. Fetch MassWiki Hits for Unconfirmed Spectra

Requires a Bearer token. Set `BEARER_TOKEN` below.

This cell is skipped if the output file already exists — re-run with `FORCE_REFETCH = True` to re-fetch.

In [ ]:
BEARER_TOKEN = ""   # <-- paste token here
FORCE_REFETCH = False
FETCH_LIMIT   = None  # set to e.g. 100 for a quick test run

if os.path.exists(UNCONFIRMED_HITS) and not FORCE_REFETCH:
    print(f'Hits file already exists: {UNCONFIRMED_HITS}')
    print('Set FORCE_REFETCH=True to re-fetch.')
else:
    if not BEARER_TOKEN:
        raise ValueError('Set BEARER_TOKEN before fetching.')

    sys.path.insert(0, CODE_DIR)
    from masswiki_pipeline_with_token import (
        fetch_reference_hits_many, flatten_reference_hits
    )

    wiki_ids = unconfirmed['wiki_id'].astype(str).tolist()
    if FETCH_LIMIT:
        wiki_ids = wiki_ids[:FETCH_LIMIT]
        print(f'Test run: fetching {FETCH_LIMIT} ids')

    hits_dict, errors_df = fetch_reference_hits_many(
        wiki_ids,
        token=BEARER_TOKEN,
        primary=('binbase', False),
        secondary=('zyang2k', True),
        max_workers=8,
        rps=6.0,
    )

    hits_df = flatten_reference_hits(hits_dict)
    hits_df.to_csv(UNCONFIRMED_HITS, index=False)
    print(f'Saved {len(hits_df):,} hits to {UNCONFIRMED_HITS}')

    if len(errors_df):
        errors_df.to_csv(os.path.join(OUT_DIR, 'fetch_errors.csv'), index=False)
        print(f'Errors: {len(errors_df)} — saved to out_unconfirmed/fetch_errors.csv')

---
## 3. Load & Join

In [ ]:
hits_raw = pd.read_csv(UNCONFIRMED_HITS, low_memory=False)
print(f'Hits loaded: {len(hits_raw):,} rows')
print(f'  hit_source counts:\n{hits_raw["hit_source"].value_counts().to_string()}')
print()

# Merge with spectra
spec_sub = unconfirmed[['wiki_id', 'rt', 'precursor_mz', 'entropy']].copy()
joint = hits_raw.merge(spec_sub, on='wiki_id', how='inner')

# delta_ppm
joint['delta_ppm'] = (
    (joint['precursor_mz'] - joint['lib_precursor_mz'])
    / joint['lib_precursor_mz'] * 1e6
)

# delta_rt: two channels
joint['delta_rt'] = np.nan
anno_mask = joint['hit_source'] == 'annotation'
ref_mask  = joint['hit_source'] == 'reference'

if 'anno_delta_rt' in joint.columns:
    joint.loc[anno_mask, 'delta_rt'] = pd.to_numeric(
        joint.loc[anno_mask, 'anno_delta_rt'], errors='coerce'
    )
if 'predicted_rt_hilic' in joint.columns:
    joint.loc[ref_mask, 'delta_rt'] = (
        joint.loc[ref_mask, 'rt']
        - pd.to_numeric(joint.loc[ref_mask, 'predicted_rt_hilic'], errors='coerce')
    )

print(f'Joint table: {len(joint):,} rows across {joint["wiki_id"].nunique():,} spectra')
print(f'RT available: {joint["delta_rt"].notna().sum():,} / {len(joint):,}')

---
## 4. Build Biological Prior

Reuse the same quality-weighted BinBase prior from the annotated pipeline.

In [ ]:
ha = pd.read_csv(
    BINBASE_PATH,
    usecols=['name', 'target_type', 'fragment_of', 'ion_mode',
             'peak_gaussian_similarity', 'peak_pure', 'sample'],
    low_memory=False,
)

conf = ha[
    (ha['target_type'] == 'CONFIRMED')
    & (ha['ion_mode'] == 'NEGATIVE')
    & (~ha['name'].str.startswith('zz ', na=False))
    & (~ha['name'].str.startswith('yy ', na=False))
    & (ha['fragment_of'].isna())
].copy()

conf['peak_gaussian_similarity'] = pd.to_numeric(conf['peak_gaussian_similarity'], errors='coerce').fillna(0)
conf['peak_pure']                = pd.to_numeric(conf['peak_pure'],                errors='coerce').fillna(0)
conf['quality_weight']           = conf['peak_gaussian_similarity'] * conf['peak_pure']

study_counts  = conf.groupby('name')['sample'].nunique().rename('n_studies')
quality_sum   = conf.groupby('name')['quality_weight'].sum().rename('quality_sum')
prior_df      = pd.concat([quality_sum, study_counts], axis=1).reset_index()
prior_df['prior_raw'] = prior_df['quality_sum'] * np.sqrt(prior_df['n_studies'])

prior_total   = prior_df['prior_raw'].sum()
prior_lookup  = (prior_df.set_index('name')['prior_raw'] / prior_total).to_dict()
flat_prior    = 1.0 / (len(prior_lookup) + 1)

def prior_fn(name):
    return prior_lookup.get(name, flat_prior)

print(f'Prior built from {len(conf):,} confirmed BinBase records ({len(prior_lookup):,} unique compounds)')

---
## 5. Estimate P_novel from Full Dataset

Now that we have hits for unannotated spectra, P_novel can be estimated from the full picture.

In [ ]:
# All spectra in the run
all_wids         = set(spectra_all['wiki_id'].astype(str))
# Spectra with hits: confirmed (from confirmed pipeline) + unconfirmed (just fetched)
confirmed_hits   = pd.read_csv(CONFIRMED_HITS, low_memory=False)
hit_wids         = set(confirmed_hits['wiki_id'].astype(str)) | set(hits_raw['wiki_id'].astype(str))
no_hit_wids      = all_wids - hit_wids

p_novel = len(no_hit_wids) / len(all_wids)

print(f'Total spectra:             {len(all_wids):,}')
print(f'With any library hit:      {len(hit_wids):,}  ({len(hit_wids)/len(all_wids):.1%})')
print(f'No library hit (true):     {len(no_hit_wids):,}  ({p_novel:.1%})')
print()
print(f'P_novel (updated) = {p_novel:.4f}')
print('(Compare: old P_novel=0.783 was circular — only annotated spectra were queried)')

---
## 6. Score Unconfirmed Spectra

In [ ]:
P_M_NULL = norm.pdf(0, 0, SIGMA_BROAD)

def score_spectrum_group(g):
    H   = float(g['entropy'].iloc[0])
    lam = 1.0 - np.exp(-ALPHA * H)
    N   = len(g)

    sim = g['entropy_similarity'].fillna(0).values.clip(1e-9, 1 - 1e-9)
    p_ms2_uniform = np.full(N, 1.0 / N)
    p_ms2_soft    = np.exp(sim) / np.exp(sim).sum()
    LR_ms2        = (lam * p_ms2_soft + (1 - lam) * p_ms2_uniform) * N

    dppm  = g['delta_ppm'].fillna(0).values
    LR_m1 = norm.pdf(dppm, 0, SIGMA_M) / P_M_NULL

    LR_rt   = np.ones(N)
    drt     = g['delta_rt'].values
    src     = g['hit_source'].values
    has_rt  = ~np.isnan(drt)

    anno_mask = (src == 'annotation') & has_rt
    ref_mask  = (src == 'reference')  & has_rt
    if anno_mask.any():
        LR_rt[anno_mask] = norm.pdf(drt[anno_mask], 0, SIGMA_RT_ANNO) / P_RT_NULL
    if ref_mask.any():
        LR_rt[ref_mask]  = norm.pdf(drt[ref_mask],  0, SIGMA_RT_REF)  / P_RT_NULL

    priors = np.array([prior_fn(n) for n in g['lib_name'].fillna('')], dtype=float)
    priors = priors / priors.sum()

    unnorm   = priors * LR_ms2 * LR_m1 * LR_rt
    novel_u  = p_novel
    total    = unnorm.sum() + novel_u
    post     = unnorm / total
    P_novel_  = novel_u / total

    out = g[['wiki_id', 'lib_name', 'hit_source', 'entropy_similarity',
              'delta_ppm', 'delta_rt']].copy()
    out['LR_ms2']   = LR_ms2
    out['LR_ms1']   = LR_m1
    out['LR_rt']    = LR_rt
    out['prior']    = priors
    out['post']     = post
    out['P_novel']  = P_novel_
    out['entropy']  = H
    out['is_top']   = post == post.max()
    return out

parts = []
for wid, g in joint.groupby('wiki_id'):
    parts.append(score_spectrum_group(g))

scored = pd.concat(parts, ignore_index=True)
print(f'Scored: {scored["wiki_id"].nunique():,} spectra, {len(scored):,} candidate rows')

---
## 7. Build Review Queue

Flag spectra by reason for review.

In [ ]:
top = scored[scored['is_top']].drop_duplicates('wiki_id').copy()

# Second-best posterior per spectrum
second = (
    scored[~scored['is_top']]
    .groupby('wiki_id')['post'].max()
    .rename('post_2nd')
)
top = top.merge(second, on='wiki_id', how='left')
top['post_gap'] = top['post'] - top['post_2nd'].fillna(0)

# Known-compound flag: is the top call in BinBase at all?
known_names = set(prior_lookup.keys())
top['in_binbase'] = top['lib_name'].isin(known_names)

# Abstained spectra not in scored (P_novel >= threshold)
abstained_wids = set(joint['wiki_id'].unique()) - set(top[top['P_novel'] < NOTA_THRESH]['wiki_id'])

# ── Flag logic ────────────────────────────────────────────────────────────
flags = []

# HIGH_CONF_NOVEL: confident call on compound not in BinBase
mask = (top['P_novel'] < NOTA_THRESH) & (top['post'] > 0.7) & (~top['in_binbase'])
for _, row in top[mask].iterrows():
    flags.append({'wiki_id': row['wiki_id'], 'flag': 'HIGH_CONF_NOVEL',
                  'top_candidate': row['lib_name'], 'posterior': row['post'],
                  'P_novel': row['P_novel'], 'entropy': row['entropy'],
                  'delta_ppm': row['delta_ppm'], 'delta_rt': row['delta_rt']})

# NEAR_TIE: top two candidates within 0.05 posterior
mask = (top['P_novel'] < NOTA_THRESH) & (top['post_gap'] < 0.05) & (top['post'] > 0.1)
for _, row in top[mask].iterrows():
    flags.append({'wiki_id': row['wiki_id'], 'flag': 'NEAR_TIE',
                  'top_candidate': row['lib_name'], 'posterior': row['post'],
                  'P_novel': row['P_novel'], 'entropy': row['entropy'],
                  'delta_ppm': row['delta_ppm'], 'delta_rt': row['delta_rt']})

# RICH_NO_HIT: high entropy, model abstains
rich_abstain = scored[scored['wiki_id'].isin(abstained_wids)].drop_duplicates('wiki_id')
mask = rich_abstain['entropy'] > 1.0
for _, row in rich_abstain[mask].iterrows():
    flags.append({'wiki_id': row['wiki_id'], 'flag': 'RICH_NO_HIT',
                  'top_candidate': None, 'posterior': None,
                  'P_novel': row['P_novel'], 'entropy': row['entropy'],
                  'delta_ppm': None, 'delta_rt': None})

# MS1_RESCUE: low entropy sim but high posterior (RT/MS1 driving the call)
mask = (top['P_novel'] < NOTA_THRESH) & (top['post'] > 0.6) & (top['entropy_similarity'] < 0.5)
for _, row in top[mask].iterrows():
    flags.append({'wiki_id': row['wiki_id'], 'flag': 'MS1_RESCUE',
                  'top_candidate': row['lib_name'], 'posterior': row['post'],
                  'P_novel': row['P_novel'], 'entropy': row['entropy'],
                  'delta_ppm': row['delta_ppm'], 'delta_rt': row['delta_rt']})

# Also add: spectra with no library hit at all (unscored completely)
unscored_wids = set(unconfirmed['wiki_id'].astype(str)) - set(joint['wiki_id'].astype(str))
unscored_spec = unconfirmed[unconfirmed['wiki_id'].astype(str).isin(unscored_wids)]
for _, row in unscored_spec[unscored_spec['entropy'] > 1.0].iterrows():
    flags.append({'wiki_id': row['wiki_id'], 'flag': 'RICH_NO_HIT',
                  'top_candidate': None, 'posterior': None,
                  'P_novel': 1.0, 'entropy': row['entropy'],
                  'delta_ppm': None, 'delta_rt': None})

queue = pd.DataFrame(flags).drop_duplicates('wiki_id').sort_values(
    ['flag', 'posterior'], ascending=[True, False]
).reset_index(drop=True)

print('=== Review Queue Summary ===')
print(queue['flag'].value_counts().to_string())
print(f'\nTotal flagged: {len(queue):,}')

In [ ]:
# ── Show top examples per flag category ───────────────────────────────────
for flag in queue['flag'].unique():
    sub = queue[queue['flag'] == flag].head(5)
    print(f'\n=== {flag} (top 5) ===')
    print(sub[['wiki_id','top_candidate','posterior','entropy','delta_ppm','delta_rt']].to_string(index=False))

---
## 8. Save Outputs

In [ ]:
# Full scored table
scored.to_csv(os.path.join(OUT_DIR, 'unconfirmed_scored.csv'), index=False)

# Review queue
queue.to_csv(os.path.join(OUT_DIR, 'review_queue.csv'), index=False)

# Top calls only (one row per spectrum)
top_calls = top.copy()
top_calls['call'] = np.where(top_calls['P_novel'] >= NOTA_THRESH, 'ABSTAIN', top_calls['lib_name'])
top_calls.to_csv(os.path.join(OUT_DIR, 'unconfirmed_top_calls.csv'), index=False)

print(f'Saved to {OUT_DIR}/')
print(f'  unconfirmed_scored.csv    — {len(scored):,} rows')
print(f'  unconfirmed_top_calls.csv — {len(top_calls):,} rows')
print(f'  review_queue.csv          — {len(queue):,} rows')